# hanilsf_optimizer — 재고 수 · 유사 상품 · 배분 (노트북)

의령공장 1·2호기 상품을 특정하는 dict 를 넣으면 ERP(NEOE, **읽기 전용**)에서
**모듈1 재고 수**(`stock`), **모듈2 유사(대체) 상품 목록**(`similar_products`), 그리고 **재고출하 / 대체검토 / 생산의뢰 배분**(`check`)을 돌려준다.

위에서부터 차례로 실행한다. 저장된 출력은 **2026-09-14 실행 결과**이며 재고는 매일 바뀐다.

준비 (한 번):

```bash
pip install "git+https://github.com/koreaben777/hanil-order-check"      # hhhs-db-manager · DB 드라이버가 함께 설치된다
export HHHS_ENV_FILE=<ERP 접속정보 .env 경로>                             # 또는 이 노트북 폴더에 .env 를 둔다
```

소스 체크아웃(`dev/`)에서 `DB조회도구/.venv` 커널로 열면 설치 없이 그대로 실행된다. 자세한 입출력 설명은 `README.md`.

## 0. 준비 — import

In [1]:
import json
import pandas as pd
import hanilsf_optimizer as ho   # ERP 접속정보(.env)는 hhhs_db_manager 가 HHHS_ENV_FILE → 현재 폴더 → 설치 폴더 순으로 찾는다

print("준비 완료 · 기본 대체 기준:", ho.DEFAULT_RULE)

준비 완료 · 기본 대체 기준: {'width_plus': 200, 'length_plus': 20, 'items': [], 'grades': [], 'note': ''}


## 1. 입력 — 상품을 특정하는 dict

| 키 | 필수 | 뜻 |
| --- | --- | --- |
| `item` | ○ | 품목코드. 첫 자리 `1`=1호기 PET/PLA, `2`=2호기 PP 만 |
| `width` | ○ | 폭 mm |
| `length` | ○ | 길이 m |
| `grade` | | 등급 `A`·`A0`·`A1`·`B`·`C`·`D`·`R`. 생략하면 `A` |

이 네 키 외의 키는 무시한다. 수량·거래처는 상품 스펙이 아니므로 여기 넣지 않는다(배분 `check` 에서만 `rolls`/`kg`/`partner`).

In [2]:
p = {"item": "2PD2030WH2N", "width": 1000, "length": 1000, "grade": "A"}   # 2호기 PP · 30 g/㎡ · WHITE · 1000mm × 1000m · A등급 (TESTING.md T3)
p

{'item': '2PD2030WH2N', 'width': 1000, 'length': 1000, 'grade': 'A'}

## 2. 모듈1 — `stock(p)` 재고 수

| 키 | 뜻 |
| --- | --- |
| `rolls` `kg` | 현재고 (의령 SB창고 잔량>0 LOT 수·중량, ERP 수주화면 "현재고"와 같은 기준) |
| `open_rolls` | 미출하 수주에 이미 잡힌 롤 (앞선 주문이 선점) |
| `avail_rolls` | `rolls − open_rolls` — **지금 출하에 쓸 수 있는 롤**. 재고 유무는 이 값으로 본다 |

없는 규격이면 모두 0, 없는 품목코드면 `ValueError`.

In [3]:
s = ho.stock(p)
print(json.dumps(s, ensure_ascii=False, indent=2))

{
  "item": "2PD2030WH2N",
  "width": 1000,
  "length": 1000,
  "grade": "A",
  "rolls": 49,
  "kg": 1470.0,
  "open_rolls": 9,
  "avail_rolls": 40
}


## 3. 모듈2 — `similar_products(p, n)` 유사(대체) 상품 목록

동일규격 대신 내보낼 수 있는 후보를 최대 `n`개, **손실 적은 순**(같은 품목 → 같은 등급 → 좁은 폭 → 짧은 길이)으로 돌려준다.
기본 범위는 같은 품목·등급에서 **폭 +200mm · 길이 +20m** 까지(넓은 폭은 슬리팅, 긴 길이는 재단 가정). 동일규격과 가용 0 인 규격은 뺀다.
`kind` 가 무엇이 다른지(`폭+30`, `길이+150`, `등급`, `품목`)를 보여준다. 빈 목록이면 기준 안에 대체 가능한 재고가 없다는 뜻.

In [4]:
sim = ho.similar_products(p, n=10)
print(json.dumps(sim, ensure_ascii=False, indent=2))

[
  {
    "item": "2PD2030WH2N",
    "width": 1050,
    "length": 1000,
    "grade": "A",
    "kind": "폭+50",
    "rolls": 9,
    "kg": 283.5,
    "open_rolls": 0,
    "avail_rolls": 9
  },
  {
    "item": "2PD2030WH2N",
    "width": 1090,
    "length": 1000,
    "grade": "A",
    "kind": "폭+90",
    "rolls": 6,
    "kg": 196.2,
    "open_rolls": 0,
    "avail_rolls": 6
  }
]


### 3-1. 허용 범위를 넓혀서 — 이 호출에만 쓰는 임시 기준 `rule=`

거래처별로 저장된 기준은 `partner="거래처코드"` 로 적용한다(`substitute_rules.json`, 영업담당자 입력). 아래는 저장 없이 폭 +300 · 길이 +500 · 등급 A1 까지 넓힌 예.

In [5]:
wide = ho.similar_products(p, n=10, rule={"width_plus": 300, "length_plus": 500, "grades": ["A1"]})
pd.DataFrame(wide) if wide else "후보 없음"

,item,width,length,grade,kind,rolls,kg,open_rolls,avail_rolls
0,2PD2030WH2N,1050,1000,A,폭+50,9,283.5,0,9
1,2PD2030WH2N,1090,1000,A,폭+90,6,196.2,0,6
2,2PD2030WH2N,1300,1000,A,폭+300,25,975.0,0,25


## 4. 배분 — `check(주문)` 재고출하 / 대체검토 / 생산의뢰

상품 dict 에 수량(`rolls` 또는 `kg`)을 더해 넣으면 주문 롤수를 재고출하 → 대체검토 → 생산의뢰 순으로 배정한다. 세 경로에 동시에 걸릴 수 있다.

In [6]:
r = ho.check({**p, "rolls": 60})            # 수량은 rolls 또는 kg. 거래처 기준을 쓰려면 "partner": "거래처코드"
print(ho.report(r))

주문   2PD2030WH2N PP WHITE 30g WH/N | 1000mm × 1000m | A | 60롤 = 1,800.0 kg (롤당 30.0 kg)
현재고 동일규격 49롤 / 1,470.0 kg  −  미출하수주 9롤 (2건)  =  가용 40롤   [원장 20260000 이후, 창고 3000]
판단   ▶ 재고출하 + 대체검토 + 생산의뢰
       재고출하 40롤 (67%) · 대체검토 15롤 (25%) · 생산의뢰 5롤 (8%) = 150.0 kg

대체 후보 [기본 기준: 폭 +200mm · 길이 +20m] 넓은 폭은 슬리팅·긴 길이는 재단 가정:
  [폭+50] 2PD2030WH2N 1050×1000 A  현재고 9 − 미출하 0 = 가용 9롤  → 배정 9롤
  [폭+90] 2PD2030WH2N 1090×1000 A  현재고 6 − 미출하 0 = 가용 6롤  → 배정 6롤

동일규격 미출하 수주:
  SOZ20260800138-1 20260824 잔량 90.0 kg
  SOZ20260900080-1 20260909 잔량 180.0 kg

최근 14일 생산요청(품목 단위): 10건 / 요청 11,059.8 kg, 작업 0.0 kg
  IRQ20260900132 납기 20260910 요청 1,166.4 작업 0.0
  IRQ20260900132 납기 20260910 요청 936.0 작업 0.0
  IRQ20260900132 납기 20260910 요청 480.0 작업 0.0
  IRQ20260900132 납기 20260910 요청 450.0 작업 0.0
  IRQ20260900058 납기 20260904 요청 556.2 작업 0.0


In [7]:
{k: r[k] for k in ("need_rolls", "onhand_rolls", "open_rolls", "avail_rolls", "stock_rolls", "sub_rolls", "prod_rolls", "decisions")}

{'need_rolls': 60,
 'onhand_rolls': 49,
 'open_rolls': 9,
 'avail_rolls': 40,
 'stock_rolls': 40,
 'sub_rolls': 15,
 'prod_rolls': 5,
 'decisions': ['재고출하', '대체검토', '생산의뢰']}

## 5. 입력 오류는 `ValueError` 로 한국어 메시지가 난다

In [8]:
for bad in ({"item": "3PD2060UB1N", "width": 600, "length": 200},   # 3호기 → 범위 밖
            {"item": "2PD2040NT1N", "width": 1070}):                # length 누락
    try:
        ho.stock(bad)
    except ValueError as e:
        print("ValueError:", e)

ValueError: 3PD2060UB1N: 1·2호기 품목(코드 첫 자리 1/2)만 지원합니다.
ValueError: 필수 키가 빠졌습니다: ['length'] (필요: item, width, length)


## 6. 다른 상품으로 해보기

1 의 `p` 만 바꿔 2~4 를 다시 실행한다. 품목코드는 웹앱(`app.py`)의 자동완성이나 `hhhs-db table MA_PITEM -w "CLS_ITEM='003'"` 로 찾을 수 있다.
JSON 파일로 남기려면 `json.dump(ho.jsonable(r), open("result.json", "w"), ensure_ascii=False, indent=2)`.